# 第 3 章：PyTorch 与模型框架——完整实验版
本 Notebook 追踪一个模型从训练框架进入部署 Runtime 的完整旅程。重点不是背 API，而是区分模型定义、参数状态、计算图和 Runtime 输出各自是什么。

主线：`nn.Module → state_dict → ONNX Graph → ONNX Runtime → 数值验证`。每一步都会产生不同类型的工程证据。

## 准备环境：让导出链路可复现

这里导入 PyTorch、ONNX 和 ONNX Runtime。它们分别扮演训练框架、模型表示和推理 Runtime。固定随机种子使模型参数可复现；`artifacts/` 专门保存本次实验产生的工件。

In [ ]:
from pathlib import Path
import numpy as np
import torch
from torch import nn
import onnx
import onnxruntime as ort
torch.manual_seed(42); artifacts=Path('artifacts'); artifacts.mkdir(exist_ok=True)

# 实验一：模型、参数与 Checkpoint

**为什么做？** 很多人把 `.pth` 文件误认为完整模型。这里先定义一个 `8 → 16 → 3` MLP，再只保存 `state_dict`，用新的模型实例加载它。

`state_dict` 保存参数与 Buffer，不保存 `DemoMLP` 类本身。接收方需要拥有兼容的模型定义。`eval()` 则明确模型进入推理状态，对 Dropout、BatchNorm 等层尤其重要。

**运行后观察：** 重载前后的输出误差应为 0。思考：若把隐藏层宽度从 16 改成 32，再加载同一状态会发生什么？

In [ ]:
class DemoMLP(nn.Module):
    def __init__(self):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(in_features=8, out_features=16),
            nn.ReLU(),
            nn.Linear(in_features=16, out_features=3),
        )

    def forward(self, inputs):
        return self.net(inputs)

# 创建模型和贯穿导出链路的固定输入
model = DemoMLP().eval()
sample = torch.linspace(-1, 1, 32).reshape(4, 8)

# state_dict 只保存参数状态
checkpoint_path = artifacts / 'reference_model.pth'
torch.save(model.state_dict(), checkpoint_path)

# 接收方先重建结构，再加载参数
reloaded_model = DemoMLP()
saved_state = torch.load(checkpoint_path, weights_only=True)
reloaded_model.load_state_dict(saved_state)
reloaded_model.eval()

with torch.no_grad():
    original_output = model(sample)
    reloaded_output = reloaded_model(sample)
reload_error = torch.max(torch.abs(original_output - reloaded_output)).item()
print('reload error =', reload_error)

# 实验二：导出并检查 ONNX Graph

**为什么做？** Checkpoint 仍依赖 PyTorch 模型定义；ONNX 把执行所需的计算表达为跨框架图。导出过程会把 Linear、ReLU 等框架层转换成 ONNX Operator。

样例输入不仅提供数值，也帮助导出器确定 Shape 和计算路径。我们只把第 0 维标记为动态 Batch，特征维仍固定为 8。Opset 17 表示图中算子遵循的 ONNX 版本契约。

**运行后观察：** Checker 通过说明图结构合法；打印的 `Gemm → Relu → Gemm` 展示 PyTorch 层怎样落到 ONNX 节点。合法图不等于所有硬件都支持这些节点。

In [ ]:
onnx_path = artifacts / 'reference_model.onnx'
torch.onnx.export(
    model,
    sample,
    onnx_path,
    input_names=['input'],
    output_names=['output'],
    dynamic_axes={
        'input': {0: 'batch'},
        'output': {0: 'batch'},
    },
    opset_version=17,
    dynamo=False,  # 固定使用 dynamic_axes 对应的导出路径
)

# Checker 验证格式与图结构；打印节点帮助建立层到算子的映射
onnx_model = onnx.load(onnx_path)
onnx.checker.check_model(onnx_model)
for node in onnx_model.graph.node:
    print(f'{node.op_type:10s}', node.name)

# 实验三：ONNX Runtime 与数值验证

**为什么做？** 成功导出只证明生成了文件，不能证明转换后的模型保持原语义。部署验证必须把完全相同的输入交给源框架和目标 Runtime，并比较原始输出。

这里明确指定 `CPUExecutionProvider`，避免对执行位置产生误判。`session.run` 使用导出时约定的输入/输出名称；名称、Shape、Dtype 都属于模型接口契约。

**运行后观察：** 两边输出 Shape 应相同；最大和平均绝对误差应很小但不必逐位为零。思考为什么容差需要结合模型任务、精度类型和下游决策设定。

In [ ]:
# 源框架输出：评估模式、不记录梯度
with torch.no_grad():
    pytorch_output = model(sample).numpy()

# 目标 Runtime 输出：明确指定执行提供器与接口名称
session = ort.InferenceSession(
    str(onnx_path), providers=['CPUExecutionProvider']
)
ort_output = session.run(
    ['output'], {'input': sample.numpy()}
)[0]

# 不只判断能否运行，还要量化两个实现的输出差异
absolute_error = np.abs(pytorch_output - ort_output)
print('providers =', session.get_providers())
print('output shape =', ort_output.shape)
print('max absolute error =', absolute_error.max())
print('mean absolute error =', absolute_error.mean())

# 统一验收

验收器读取当前 `model` 与 `sample`，重新加载 Checkpoint、核对模型输出、检查 ONNX 图并实际执行 ORT，验证跨 Runtime 数值一致性。它刻意不把“文件存在”当作成功：文件还必须可解析、可执行，并在约定容差内保持输出。通过后会生成机器可读的参考结果。

In [ ]:
from validate import validate_exercise,print_report,save_report
report=validate_exercise(globals()); print_report(report)
if report['passed']: save_report(report,artifacts/'reference_result.json')